# Supervised regime router → named experts

Test a different routing objective: **classify time of day, then route the whole image to the expert named by the predicted class**. The fixed mapping is **E0 = daytime, E1 = dawn/dusk, E2 = night**, independently at each MoE block. Every spatial token and CLS in that block goes to the image's predicted expert. The two blocks may disagree; both are evaluated separately.

This is a supervised routing experiment. Labels train router classifiers; inference uses image content alone. Ground-truth labels never override routing during evaluation. A dark tunnel is not automatically labeled night: its original time-of-day label is retained, and clips with undefined time of day have no regime accuracy score.

Warm-start shared weights, decoder, and three expert MLPs from the existing three-epoch MAE. The old unsupervised expert IDs had no class meaning: the new router heads are initialized afresh. Train only routers first, then optionally train routers, expert MLPs and decoder jointly using reconstruction plus classification. Shared backbone weights remain frozen by default. Original checkpoints and notebooks are never overwritten.

```powershell
uv run --extra cpu --with pandas --with matplotlib --with opencv-python-headless --with jupyterlab jupyter lab
```

Use the project's CUDA environment/kernel for a full run. Full training uses all 70,000 stills on demand. Unknown regime labels contribute reconstruction updates only. Caps below are for smoke tests and are saved in the manifest.


In [ ]:
import json
import math
import random
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cafl4ds/configs").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from inside the cafl4ds repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from collections.abc import Iterator
from contextlib import contextmanager
from functools import partial

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display as show
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

from cafl4ds.jsonio import dumps_valid
from cafl4ds.models.vit import MoEViTEncoder, patchify
from cafl4ds.ssl.factory import build_mae

## Settings

Class-weighted cross-entropy compensates for fewer dawn/dusk training images. Uniform load balancing and router noise are disabled: forcing equal expert usage would compete with these supervised, uneven class frequencies. The repository's full-softmax gate scaling is retained. Top-1 inference always uses **predicted** class IDs.

Stage 1 uses unmasked full images to match video inference. Stage 2 adds MAE reconstruction on masked views plus router classification on both full and masked views. Per-block raw logits are captured from the actual differentiable forward; the no-gradient `routing()` diagnostic is never used as a training loss.


## Dataset and train/support/test split — edit here

All split settings and construction code live in this notebook. Set `bdd_root` to your extracted BDD
directory if automatic local discovery does not find it. The notebook reads images and original BDD
attributes directly; it does not import train/test membership from a previous notebook.

- `split_mode="official"` keeps the full official training split and divides official validation
  clips using **support_fraction:test_fraction**. `train_fraction` is unused in this mode.
- `split_mode="random"` combines official train/val and assigns clips using **train_fraction,
  support_fraction, test_fraction**, which must be positive and sum to 1. For example, 0.8/0.1/0.1.
- Support images fit the linear/kNN readouts; test (`query`) images measure performance.
- Set both evaluation caps to `None` to use the full allocated support/test panels. Fractions apply
  before label filtering, sparse-stratum rounding and caps; actual counts are printed and saved.
- Splits are seeded, stratified and disjoint by **clip ID**. Known video clips are excluded first.
  Training seeds and model arms share the same split when these settings match.

Use a fresh run when changing the split. A warm-start/frozen checkpoint must have trained on disjoint
clips; those notebooks check this explicitly. Saved plots below describe their original completed
runs until you rerun the affected cells with the edited settings.


In [ ]:
# Edit the dataset and split here; no pre-generated CSVs or external configuration are needed.
bdd_root = None  # Set a local path here if automatic dataset discovery cannot find your BDD extraction.
split_mode = "official"  # "official": keep BDD train/val; "random": split the combined image pool.
train_fraction = 0.80  # Used by random mode; official mode retains all official training images.
support_fraction = 0.10  # Labeled support fits the frozen linear/kNN readouts.
test_fraction = 0.10  # Held-out test images are called query in the existing evaluation code.
split_seed = 13  # Use the same value in each notebook for paired comparisons.
split_stratify_by = ("scene", "regime")  # Add "weather", or use () for unstratified splitting.
support_per_scene = 32  # Optional cap AFTER splitting; None uses every eligible support image.
query_per_scene_regime = 64  # Optional cap AFTER splitting; None uses every eligible test image.
extra_excluded_clip_ids = []  # Additional video/sequence IDs to reserve from all still-image partitions.
video_exclusion_manifests = [
    "outputs/bdd-illumination/selected_videos.json",
    "outputs/bdd-day-dusk-night/matched_sources.json",
    "outputs/bdd-video-subset/video_manifest.json",
]

In [ ]:
# Split construction is embedded here; this notebook does not import an external split/config script.
import csv as split_csv
import hashlib as split_hashlib
import json as split_json
import math as split_math
from collections import defaultdict as split_defaultdict


def valid_bdd_root(path: Path) -> bool:
    """Recognize the extracted BDD still-image layout without requiring a previous experiment."""
    return all((path / "images/100k" / name).is_dir() for name in ["train", "val"])


def resolve_bdd_root(configured: str | Path | None) -> Path:
    """Use the notebook's path or discover a local dataset; saved CSVs supply path hints only."""
    if configured is not None:
        candidate = Path(configured).expanduser()
        candidate = candidate if candidate.is_absolute() else repo_root / candidate
        if not valid_bdd_root(candidate):
            raise FileNotFoundError(f"Expected images/100k/train and val below bdd_root={candidate}.")
        return candidate.resolve()
    candidates = [repo_root / name for name in ["data/bdd100k", "data", "bdd100k", "datasets/bdd100k"]]
    for family in ["moe-stills-to-video", "moe-router-comparison", "moe-router-illumination-comparison"]:
        for filename in ["ssl_train.csv", "train.csv"]:
            for manifest in sorted((repo_root / "outputs" / family).glob(f"*/{filename}"), reverse=True):
                with manifest.open(newline="", encoding="utf-8") as stream:
                    first = next(split_csv.DictReader(stream), None)
                if first and first.get("path"):
                    candidates.extend(Path(first["path"]).parents)
    for candidate in candidates:
        if valid_bdd_root(candidate):
            return candidate.resolve()
    raise FileNotFoundError("Set bdd_root in this notebook to your extracted BDD image/label directory.")


def read_bdd_attributes(folder: Path, split: str) -> dict:
    """Read combined or per-image official attributes directly from the configured dataset."""
    combined = folder / "labels" / f"bdd100k_labels_images_{split}.json"
    individual = folder / "labels/100k" / split
    if combined.is_file():
        items = split_json.loads(combined.read_text(encoding="utf-8"))
    elif individual.is_dir():
        items = [split_json.loads(p.read_text(encoding="utf-8")) for p in sorted(individual.glob("*.json"))]
    else:
        raise FileNotFoundError(f"Missing BDD attributes: {combined} or {individual}")
    return {Path(item["name"]).stem: item.get("attributes", {}) for item in items}


def read_bdd_stills(folder: Path) -> list[dict]:
    """Read paths and attributes, without decoding images or loading a saved split."""
    rows = []
    for split in ["train", "val"]:
        attributes = read_bdd_attributes(folder, split)
        for path in sorted((folder / "images/100k" / split).iterdir()):
            if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            attrs = attributes.get(path.stem, {})
            rows.append(
                {
                    "path": str(path),
                    "clip_id": path.stem,
                    "split": split,
                    "scene": attrs.get("scene") or "undefined",
                    "regime": attrs.get("timeofday") or "unknown",
                    "weather": attrs.get("weather") or "undefined",
                }
            )
    return rows


def reserved_video_ids() -> set[str]:
    """Exclude every configured evaluation-video clip before assigning still-image partitions."""
    result = set(extra_excluded_clip_ids)
    for relative in video_exclusion_manifests:
        path = repo_root / relative
        if path.is_file():
            entries = split_json.loads(path.read_text(encoding="utf-8"))
            result.update(str(row["clip_id"]) for row in entries)
    return result


def split_counts(size: int, fractions: list[float]) -> list[int]:
    """Use largest-remainder rounding; retain each partition in sufficiently large strata."""
    expected = [size * fraction for fraction in fractions]
    counts = [split_math.floor(value) for value in expected]
    order = sorted(range(len(counts)), key=lambda i: (expected[i] - counts[i], fractions[i]), reverse=True)
    for index in order[: size - sum(counts)]:
        counts[index] += 1
    if size >= len(counts):
        for index, count in enumerate(counts):
            if count == 0:
                donor = max(range(len(counts)), key=lambda i: counts[i] - expected[i])
                if counts[donor] <= 1:
                    donor = max(range(len(counts)), key=lambda i: counts[i])
                counts[donor] -= 1
                counts[index] += 1
    return counts


def partition_clips(rows: list[dict], names: list[str], fractions: list[float]) -> dict:
    """Assign entire clips using stable seeded ordering within the requested metadata strata."""
    clips = split_defaultdict(list)
    for row in rows:
        clips[str(row["clip_id"])].append(row)
    strata = split_defaultdict(list)
    for clip_id, members in clips.items():
        keys = {tuple(str(row.get(key, "unknown")) for key in split_stratify_by) for row in members}
        if len(keys) != 1:
            raise ValueError(f"Conflicting stratification attributes within clip {clip_id}.")
        strata[next(iter(keys))].append(clip_id)
    partitions = {name: [] for name in names}
    for key in sorted(strata):
        ordered = sorted(
            strata[key], key=lambda clip: split_hashlib.sha256(f"{split_seed}|{clip}".encode()).hexdigest()
        )
        counts = split_counts(len(ordered), fractions)
        start = 0
        for name, count in zip(names, counts, strict=True):
            partitions[name].extend(row for clip in ordered[start : start + count] for row in clips[clip])
            start += count
    return partitions


def cap_split_groups(rows: list[dict], keys: tuple[str, ...], cap: int | None) -> list[dict]:
    """Apply optional per-group evaluation caps after splitting, retaining whole clips."""
    if cap is None:
        return rows
    if not isinstance(cap, int) or isinstance(cap, bool) or cap < 1:
        raise ValueError("Evaluation caps must be positive integers or None.")
    groups = split_defaultdict(dict)
    for row in rows:
        key = tuple(row[name] for name in keys)
        groups[key].setdefault(row["clip_id"], []).append(row)
    return [row for key in sorted(groups) for members in list(groups[key].values())[:cap] for row in members]


def validate_split_settings() -> list[float]:
    """Validate fractions and stratification before reading any dataset metadata."""
    fractions = [train_fraction, support_fraction, test_fraction]
    if any(not split_math.isfinite(value) or value <= 0 for value in fractions):
        raise ValueError("train_fraction, support_fraction and test_fraction must be finite and positive.")
    if not split_math.isclose(sum(fractions), 1.0, abs_tol=1e-9):
        raise ValueError("train_fraction + support_fraction + test_fraction must equal 1.")
    if split_mode not in {"official", "random"}:
        raise ValueError('split_mode must be "official" or "random".')
    if not set(split_stratify_by).issubset({"scene", "regime", "weather"}):
        raise ValueError("Stratification may use scene, regime, and/or weather; () gives unstratified splitting.")
    return fractions


def build_notebook_splits(extra_video_ids: set[str] | None = None) -> tuple[dict, list[str], dict]:
    """Build train/support/test directly from BDD metadata using only this notebook's settings."""
    fractions = validate_split_settings()
    excluded = reserved_video_ids() | set(extra_video_ids or ())
    all_rows = read_bdd_stills(dataset_root)
    rows = [row for row in all_rows if row["clip_id"] not in excluded]
    if split_mode == "official":
        train = [row for row in rows if row["split"] == "train"]
        evaluation = [row for row in rows if row["split"] == "val"]
        ratio = support_fraction / (support_fraction + test_fraction)
        parts = {"train": train, **partition_clips(evaluation, ["support", "query"], [ratio, 1 - ratio])}
    else:
        parts = partition_clips(rows, ["train", "support", "query"], fractions)
    for name in ["support", "query"]:
        parts[name] = [
            row
            for row in parts[name]
            if row["scene"] not in {"undefined", ""} and row["regime"] in {"daytime", "dawn/dusk", "night"}
        ]
    parts["support"] = cap_split_groups(parts["support"], ("scene",), support_per_scene)
    parts["query"] = cap_split_groups(parts["query"], ("scene", "regime"), query_per_scene_regime)
    scenes = sorted({row["scene"] for row in parts["support"]} & {row["scene"] for row in parts["query"]})
    for name in ["support", "query"]:
        parts[name] = [row for row in parts[name] if row["scene"] in scenes]
    if any(not part for part in parts.values()) or len(scenes) < 2:
        raise ValueError("Split is empty or has fewer than two shared scene classes; increase data/fractions or caps.")
    ids = {name: {row["clip_id"] for row in part} for name, part in parts.items()}
    if ids["train"] & (ids["support"] | ids["query"]) or ids["support"] & ids["query"]:
        raise ValueError("Clip overlap between train/support/test; check duplicate BDD source identifiers.")
    contract = {
        "version": "notebook-bdd-clip-split-v1",
        "dataset_root": str(dataset_root),
        "mode": split_mode,
        "train_fraction": train_fraction,
        "support_fraction": support_fraction,
        "test_fraction": test_fraction,
        "seed": split_seed,
        "stratify_by": list(split_stratify_by),
        "support_per_scene": support_per_scene,
        "query_per_scene_regime": query_per_scene_regime,
        "excluded_video_ids_sha256": split_hashlib.sha256("\n".join(sorted(excluded)).encode()).hexdigest(),
    }
    print("Split built in this notebook:", {name: len(part) for name, part in parts.items()})
    if split_mode == "official":
        print(
            "Official train retained; official val divided by support_fraction:test_fraction; train_fraction is unused."
        )
    print("Fractions apply before unknown-label filtering, sparse-stratum rounding and evaluation caps.")
    return parts, scenes, contract


def guard_notebook_split(folder: Path, contract: dict) -> None:
    """Reject edited split settings before overwriting manifests in an existing run."""
    path = folder / "split_settings.json"
    if path.exists() and split_json.loads(path.read_text()) != contract:
        raise ValueError("Split settings changed. Choose a fresh run directory before regenerating its partitions.")
    path.write_text(split_json.dumps(contract, indent=2), encoding="utf-8")


def find_stills_checkpoint(configured: str | Path | None) -> Path:
    """Find an existing model for notebooks that explicitly perform warm-start or frozen evaluation."""
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else sorted((repo_root / "outputs/moe-stills-to-video").glob("*/checkpoint.pt"), reverse=True)
    )
    for candidate in candidates:
        resolved = candidate if candidate.is_absolute() else repo_root / candidate
        folder = resolved.parent if resolved.is_file() else resolved
        if all((folder / name).is_file() for name in ["checkpoint.pt", "config.json", "ssl_train.csv"]):
            return folder.resolve()
    raise FileNotFoundError("Set checkpoint_run_dir in this notebook to a trained stills-to-video model run.")


def check_pretrained_split(folder: Path, parts: dict) -> None:
    """A warm-start/frozen checkpoint must never have pretrained on the new evaluation clips."""
    with (folder / "ssl_train.csv").open(newline="", encoding="utf-8") as stream:
        trained = {row["clip_id"] for row in split_csv.DictReader(stream)}
    overlap = trained & {row["clip_id"] for name in ["support", "query"] for row in parts[name]}
    overlap |= trained & reserved_video_ids()
    if overlap:
        raise ValueError(
            f"The selected checkpoint trained on {len(overlap)} new support/test clips. "
            "Train moe_stills_to_video with these same split settings first, then select that checkpoint_run_dir."
        )


dataset_root = resolve_bdd_root(bdd_root)

In [ ]:
checkpoint_run_dir = None  # Optional model run path; split settings remain entirely above.
source_run = find_stills_checkpoint(checkpoint_run_dir)
regime_names = ["daytime", "dawn/dusk", "night"]
regime_to_id = {name: index for index, name in enumerate(regime_names)}
seed = 13
batch_size = 64
router_epochs = 2
expert_epochs = 1
router_learning_rate = 1e-3
joint_learning_rate = 1e-4
classification_weight = 1.0
masked_classification_weight = 0.5
gradient_clip_norm = 1.0
freeze_shared_backbone = True
max_train_images = None
max_steps_per_epoch = None
max_validation_images = None
sample_every_seconds = 0.5
max_video_samples = None
prefer_cuda = True

if min(router_epochs, expert_epochs) < 0 or router_epochs + expert_epochs < 1:
    raise ValueError("Choose nonnegative stage lengths and at least one training epoch.")
if min(router_learning_rate, joint_learning_rate, gradient_clip_norm, classification_weight) <= 0:
    raise ValueError("Learning rates, gradient clipping and classification weight must be positive.")
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
device = torch.device("cuda" if prefer_cuda and torch.cuda.is_available() else "cpu")
run_dir = repo_root / "outputs/regime-router" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
source_config = json.loads((source_run / "config.json").read_text(encoding="utf-8"))
source_checkpoint = torch.load(source_run / "checkpoint.pt", map_location="cpu", weights_only=True)
configured_videos = json.loads((repo_root / "outputs/bdd-illumination/selected_videos.json").read_text())
twilight = json.loads((repo_root / "outputs/bdd-video-subset/video_manifest.json").read_text())
for scene in ["city street", "highway", "residential"]:
    configured_videos.append(next(spec for spec in twilight if spec["scene"] == scene))
video_ids = {spec["clip_id"] for spec in configured_videos}
print("Expert labels:", dict(enumerate(regime_names)))
print(f"Device: {device}; output directory: {run_dir}")

In [ ]:
split_parts, scene_names, split_contract = build_notebook_splits(video_ids)
check_pretrained_split(source_run, split_parts)
guard_notebook_split(run_dir, split_contract)
train_frame, support_frame, query_frame = [pd.DataFrame(split_parts[name]) for name in ["train", "support", "query"]]
if max_train_images is not None:
    # A deterministic stratified cap keeps all three classes in a smoke test.
    groups = [part.sample(frac=1, random_state=seed) for _, part in train_frame.groupby("regime", dropna=False)]
    train_frame = pd.concat([part.head(max(1, max_train_images // len(groups))) for part in groups]).head(
        max_train_images
    )
if max_validation_images is not None:
    groups = [part for _, part in query_frame.groupby("regime")]
    query_frame = pd.concat([part.head(max(1, max_validation_images // len(groups))) for part in groups]).head(
        max_validation_images
    )
train_records = train_frame.to_dict("records")
support_records = support_frame.to_dict("records")
query_records = query_frame.to_dict("records")
sets = {
    name: {record["clip_id"] for record in records}
    for name, records in [("train", train_records), ("support", support_records), ("query", query_records)]
}
if sets["train"] & (sets["support"] | sets["query"]) or sets["support"] & sets["query"]:
    raise ValueError("Still-image partitions overlap by clip ID.")
if any(ids & video_ids for ids in sets.values()):
    raise ValueError("Video clips overlap a still-image partition.")
class_counts = np.array([(train_frame["regime"] == name).sum() for name in regime_names])
if (class_counts == 0).any():
    raise ValueError(f"Training must include all regime classes: {dict(zip(regime_names, class_counts, strict=True))}")
class_weights = torch.tensor(
    class_counts.sum() / (len(regime_names) * class_counts), dtype=torch.float32, device=device
)
scene_to_id = {name: index for index, name in enumerate(scene_names)}
knn_k = source_config["knn_k"]
for name, records in [("train", train_records), ("support", support_records), ("query", query_records)]:
    pd.DataFrame(records).to_csv(run_dir / f"{name}.csv", index=False)
show(
    pd.DataFrame(
        {
            "regime": regime_names,
            "training_images": class_counts,
            "CE_weight": class_weights.cpu().numpy(),
            "expert": range(len(regime_names)),
        }
    )
)
print("Unlabeled training images (reconstruction only):", int((~train_frame["regime"].isin(regime_names)).sum()))

In [ ]:
model_config = dict(source_checkpoint["model_config"])
model_config.update(
    num_experts=3, top_k=1, routing_level="image", router_noise_std=0.0, importance_weight=0.0, load_weight=0.0
)
mae_config = dict(source_checkpoint["mae_config"])
mae_config["aux_weight"] = 0.0
method = build_mae(MoEViTEncoder(**model_config), **mae_config).to(device)
destination = method.state_dict()
warm_state = {
    name: value
    for name, value in source_checkpoint["method"].items()
    if ".router." not in name and name in destination and value.shape == destination[name].shape
}
missing, unexpected = method.load_state_dict(warm_state, strict=False)
if unexpected or any(".router." not in name for name in missing):
    raise ValueError(f"Unexpected warm-start mismatch: missing={missing}, unexpected={unexpected}")
img_size, patch_size = model_config["img_size"], model_config["patch_size"]
num_experts, top_k, moe_blocks = 3, 1, tuple(model_config["moe_blocks"])
mask_ratio = mae_config["mask_ratio"]
fixed_mask_noise = torch.load(source_run / "reconstruction_masks.pt", map_location="cpu", weights_only=True)
if mae_config["norm_pix_loss"]:
    raise ValueError("RGB MSE evaluation expects the existing unnormalized-pixel checkpoint.")
manifest = {
    "source_run": str(source_run),
    "split_settings": split_contract,
    "source_checkpoint_epoch": source_checkpoint["epoch"],
    "regime_to_expert": regime_to_id,
    "model_config": model_config,
    "mae_config": mae_config,
    "router_epochs": router_epochs,
    "expert_epochs": expert_epochs,
    "batch_size": batch_size,
    "router_learning_rate": router_learning_rate,
    "joint_learning_rate": joint_learning_rate,
    "classification_weight": classification_weight,
    "masked_classification_weight": masked_classification_weight,
    "freeze_shared_backbone": freeze_shared_backbone,
    "class_counts": class_counts.tolist(),
    "class_weights": class_weights.cpu().tolist(),
    "max_train_images": max_train_images,
    "max_steps_per_epoch": max_steps_per_epoch,
    "max_validation_images": max_validation_images,
    "seed": seed,
    "device": str(device),
    "videos": configured_videos,
    "note": "Supervised image routing; predicted gates at train and test; no uniform balancing or router noise.",
}
(run_dir / "config.json").write_text(dumps_valid(manifest), encoding="utf-8")
print("Warm-started shared weights, decoder and expert MLPs 0–2; router heads are new.")

In [ ]:
class StillDataset(Dataset):
    """Lazy RGB loading with the exact same spatial preprocessing for stills and video."""

    def __init__(self, records: list[dict[str, Any]]) -> None:
        """Retain only paths and metadata."""
        self.records = records

    def __len__(self) -> int:
        """Count available still images."""
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        """Decode one image and return its optional scene ID."""
        record = self.records[index]
        with Image.open(record["path"]) as image:
            tensor = preprocess_rgb(np.array(image.convert("RGB")))
        return tensor, scene_to_id.get(record.get("scene"), -1)


def preprocess_rgb(rgb: np.ndarray) -> torch.Tensor:
    """RGB uint8 → float [0,1], resized with antialiasing to the model's square input."""
    tensor = TF.to_tensor(rgb)
    return TF.resize(tensor, [img_size, img_size], antialias=True)


def make_loader(records: list[dict[str, Any]], shuffle: bool = False, epoch: int = 0) -> DataLoader:
    """Use zero workers for portable Windows/Jupyter behavior and a reproducible epoch shuffle."""
    return DataLoader(
        StillDataset(records),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(seed + epoch),
        pin_memory=device.type == "cuda",
    )


@torch.no_grad()
def reconstruction_mse(images: torch.Tensor) -> torch.Tensor:
    """Return one masked RGB MSE per image, averaged over fixed mask templates."""
    target = patchify(images, patch_size)
    errors = []
    for noise in fixed_mask_noise:
        result = method.encoder.forward_with_aux(
            images, mask_ratio=mask_ratio, mask_noise=noise[None].expand(len(images), -1).to(device)
        )
        prediction = method.decoder(result.tokens, result.ids_restore)
        patch_error = (prediction - target).square().mean(-1)
        errors.append((patch_error * result.mask).sum(-1) / result.mask.sum(-1).clamp_min(1))
    return torch.stack(errors).mean(0)


@torch.no_grad()
def routing_arrays(images: torch.Tensor) -> dict[int, dict[str, np.ndarray]]:
    """Return clean spatial-token probabilities, dispatch shares, and gate weights."""
    output = {}
    for block, probabilities in method.encoder.routing(images).items():
        spatial = probabilities[:, 1:, :]  # CLS is excluded from spatial-token summaries.
        selected = F.one_hot(spatial.topk(top_k, dim=-1).indices, num_experts).sum(-2)
        output[block] = {
            "probability": spatial.mean(1).cpu().numpy(),
            "dispatch_share": (selected.float().mean(1) / top_k).cpu().numpy(),
            "selected_weight": (spatial * selected).mean(1).cpu().numpy(),
        }
    return output


@torch.no_grad()
def encode_records(records: list[dict[str, Any]]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Read ordered embeddings, labels, and masked reconstruction errors without updating weights."""
    features, labels, errors = [], [], []
    method.eval()
    for cpu_images, targets in make_loader(records):
        images = cpu_images.to(device)
        features.append(method.encode(images).cpu().numpy())
        labels.append(targets.numpy())
        errors.append(reconstruction_mse(images).cpu().numpy())
    if not features:
        raise ValueError("Empty evaluation partition.")
    return np.concatenate(features), np.concatenate(labels), np.concatenate(errors)


def fit_scene_readouts() -> tuple[Any, Any]:
    """Fit only linear/kNN readouts on validation supports, with fixed hyperparameters."""
    features, labels, _ = encode_records(support_records)
    linear = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=seed))
    knn = KNeighborsClassifier(n_neighbors=min(knn_k, len(labels)), metric="cosine", weights="distance")
    return linear.fit(features, labels), knn.fit(features, labels)


def evaluate_stills(stage: str) -> tuple[list[dict[str, Any]], Any, Any]:
    """Score disjoint query clips by regime with a fresh frozen-encoder readout."""
    linear, knn = fit_scene_readouts()
    features, labels, errors = encode_records(query_records)
    predictions = {"linear": linear.predict(features), "knn": knn.predict(features)}
    query_frame = pd.DataFrame(query_records)
    rows = []
    for regime in ["daytime", "night", "dawn/dusk"]:
        indices = np.flatnonzero(query_frame["regime"].to_numpy() == regime)
        if not len(indices):
            continue
        row = {
            "stage": stage,
            "regime": regime,
            "images": len(indices),
            "masked_rgb_mse": float(errors[indices].mean()),
        }
        for name, prediction in predictions.items():
            row[f"{name}_accuracy"] = accuracy_score(labels[indices], prediction[indices])
            row[f"{name}_macro_f1"] = f1_score(labels[indices], prediction[indices], average="macro", zero_division=0)
        rows.append(row)
    return rows, linear, knn

In [ ]:
class RegimeDataset(StillDataset):
    """Load images lazily and return a regime ID; unknown labels use -100."""

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        """Decode one image without ever passing its label into the model forward."""
        image, _scene = super().__getitem__(index)
        return image, regime_to_id.get(self.records[index].get("regime"), -100)


def regime_loader(records: list[dict[str, Any]], shuffle: bool = False, epoch: int = 0) -> DataLoader:
    """Use every available image once per epoch, with an isolated shuffle generator."""
    return DataLoader(
        RegimeDataset(records),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(seed + epoch),
        pin_memory=device.type == "cuda",
    )


@contextmanager
def capture_router_logits() -> Iterator[dict[int, torch.Tensor]]:
    """Capture graph-connected raw logits from the actual per-image gates; remove hooks afterward."""
    logits = {}

    def record(block: int, _module: torch.nn.Module, _args: tuple, output: torch.Tensor) -> None:
        """Record the logits before softmax/top-1; no detached diagnostic values."""
        logits[block] = output

    handles = [
        method.encoder.blocks[block].mlp.router.register_forward_hook(partial(record, block)) for block in moe_blocks
    ]
    try:
        yield logits
    finally:
        for handle in handles:
            handle.remove()


def classification_loss(logits: dict[int, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
    """Average class-weighted CE across the actual block routers; safely skip unlabeled batches."""
    valid = targets >= 0
    if not valid.any():
        return sum(value.sum() * 0 for value in logits.values())
    return torch.stack(
        [F.cross_entropy(value[valid], targets[valid], weight=class_weights) for value in logits.values()]
    ).mean()


def set_training_stage(stage: str) -> list[torch.nn.Parameter]:
    """Train router heads first, then add expert MLPs and decoder; shared weights stay frozen by default."""
    method.requires_grad_(False)
    for block in moe_blocks:
        layer = method.encoder.blocks[block].mlp
        layer.router.requires_grad_(True)
        layer.experts.requires_grad_(stage == "joint")
    if stage == "joint":
        method.decoder.requires_grad_(True)
        if not freeze_shared_backbone:
            method.encoder.requires_grad_(True)
    return [parameter for parameter in method.parameters() if parameter.requires_grad]


@torch.no_grad()
def evaluate_regimes(stage: str, epoch: int) -> tuple[list[dict[str, Any]], dict[int, np.ndarray]]:
    """Score actual unmasked predicted gates on disjoint validation clips."""
    modes = {module: module.training for module in method.modules()}
    method.eval()
    truths, predicted, errors = [], {block: [] for block in moe_blocks}, []
    try:
        for cpu_images, targets in regime_loader(query_records):
            images = cpu_images.to(device)
            truths.extend(targets.tolist())
            for block, probability in method.encoder.routing(images).items():
                predicted[block].extend(probability[:, 0, :].argmax(-1).cpu().tolist())
            errors.extend(reconstruction_mse(images).cpu().tolist())
    finally:
        for module, mode in modes.items():
            module.training = mode
    truths = np.array(truths)
    valid = truths >= 0
    if not valid.any():
        raise ValueError("No labeled validation regimes remain.")
    rows, matrices = [], {}
    for block, guess_list in predicted.items():
        guesses = np.array(guess_list)
        matrices[block] = confusion_matrix(truths[valid], guesses[valid], labels=range(num_experts))
        row = {
            "stage": stage,
            "epoch": epoch,
            "block": block,
            "images": int(valid.sum()),
            "regime_accuracy": accuracy_score(truths[valid], guesses[valid]),
            "balanced_accuracy": balanced_accuracy_score(truths[valid], guesses[valid]),
            "macro_f1": f1_score(
                truths[valid], guesses[valid], labels=range(num_experts), average="macro", zero_division=0
            ),
            "masked_rgb_mse": float(np.mean(errors)),
        }
        for expert, name in enumerate(regime_names):
            row[f"predicted_share_{name}"] = float((guesses[valid] == expert).mean())
        rows.append(row)
    return rows, matrices

## Train the classifying routers, then adapt the experts

Stage 1 changes only the router weights; experts, shared backbone and decoder are frozen. Stage 2 optimizes **RGB reconstruction + full-image regime CE + masked-view regime CE**. There is no oracle routing: an expert receives the samples predicted for its class, including classification mistakes. Both class accuracy and actual load are logged so a one-expert collapse is visible.

Validation is run each epoch with fixed masks, restored module modes, and no optimizer updates. These are development measurements, not an untouched test-set claim. Lower CE/MSE is better; higher accuracy/F1 is better. Scene probing remains separate from the regime loss.


In [ ]:
router_metrics, _ = evaluate_regimes("before_supervision", 0)
scene_before, _, _ = evaluate_stills("before_supervision")
history = []
global_epoch = 0
show(pd.DataFrame(router_metrics))
stages = [("router", router_epochs, router_learning_rate), ("joint", expert_epochs, joint_learning_rate)]
for stage, count, rate in stages:
    parameters = set_training_stage(stage)
    optimizer = torch.optim.AdamW(parameters, lr=rate, weight_decay=1e-4)
    for _ in range(count):
        global_epoch += 1
        method.train()
        totals = {
            "loss": 0.0,
            "full_regime_ce": 0.0,
            "masked_regime_ce": 0.0,
            "rgb_reconstruction": 0.0,
            "images": 0,
            "updates": 0,
        }
        for step, (cpu_images, cpu_targets) in enumerate(
            regime_loader(train_records, shuffle=True, epoch=global_epoch)
        ):
            if max_steps_per_epoch is not None and step >= max_steps_per_epoch:
                break
            images, targets = cpu_images.to(device), cpu_targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            with capture_router_logits() as full_logits:
                method.encoder.forward_with_aux(images, mask_ratio=0.0)
            full_ce = classification_loss(full_logits, targets)
            masked_ce, rgb_loss = images.new_zeros(()), images.new_zeros(())
            if stage == "joint":
                with capture_router_logits() as masked_logits:
                    losses = method.loss_components(images)
                masked_ce = classification_loss(masked_logits, targets)
                rgb_loss = losses.reconstruction
            total = classification_weight * full_ce + masked_classification_weight * masked_ce + rgb_loss
            if not torch.isfinite(total):
                raise FloatingPointError(f"Nonfinite loss in {stage} epoch {global_epoch}, step {step}.")
            total.backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(parameters, gradient_clip_norm, error_if_nonfinite=True)
            optimizer.step()
            for key, value in [
                ("loss", total),
                ("full_regime_ce", full_ce),
                ("masked_regime_ce", masked_ce),
                ("rgb_reconstruction", rgb_loss),
            ]:
                totals[key] += float(value.detach()) * len(images)
            totals["images"] += len(images)
            totals["updates"] += 1
            if (step + 1) % 100 == 0:
                running_ce = totals["full_regime_ce"] / totals["images"]
                print(f"{stage} epoch {global_epoch}, step {step + 1}: regime CE {running_ce:.4f}")
        if not totals["updates"]:
            raise ValueError("No training updates were performed.")
        history.append(
            {
                "stage": stage,
                "epoch": global_epoch,
                **{key: value / totals["images"] for key, value in totals.items() if key not in {"images", "updates"}},
                "images": totals["images"],
                "updates": totals["updates"],
                "last_gradient_norm": float(gradient_norm),
            }
        )
        rows, _ = evaluate_regimes(stage, global_epoch)
        router_metrics.extend(rows)
        pd.DataFrame(history).to_csv(run_dir / "training.csv", index=False)
        pd.DataFrame(router_metrics).to_csv(run_dir / "regime_metrics.csv", index=False)
        torch.save(
            {
                "method": {key: value.detach().cpu() for key, value in method.state_dict().items()},
                "model_config": model_config,
                "mae_config": mae_config,
                "regime_to_expert": regime_to_id,
                "epoch": global_epoch,
                "stage": stage,
                "history": history,
            },
            run_dir / "supervised_router.pt",
        )
        show(pd.DataFrame(rows))
method.eval()
final_rows, confusion = evaluate_regimes("final", global_epoch)
scene_after, linear_readout, knn_readout = evaluate_stills("after_supervision")
scene_metrics = pd.DataFrame(scene_before + scene_after)
scene_metrics.to_csv(run_dir / "scene_probe_metrics.csv", index=False)
show(scene_metrics)

In [ ]:
fig, axes = plt.subplots(1, len(moe_blocks), figsize=(6 * len(moe_blocks), 5), squeeze=False, constrained_layout=True)
for ax, block in zip(axes[0], moe_blocks, strict=True):
    matrix = confusion[block]
    normal = matrix / matrix.sum(1, keepdims=True).clip(min=1)
    image = ax.imshow(normal, cmap="Blues", vmin=0, vmax=1)
    for row in range(num_experts):
        for column in range(num_experts):
            ax.text(column, row, f"{normal[row, column]:.0%}\n(n={matrix[row, column]})", ha="center", va="center")
    ax.set_xticks(range(num_experts), regime_names, rotation=30)
    ax.set_yticks(range(num_experts), regime_names)
    ax.set(title=f"Block {block}: regime → selected expert", xlabel="Predicted expert class", ylabel="True regime")
fig.colorbar(image, ax=axes.ravel().tolist(), label="Fraction within true class")
fig.savefig(run_dir / "regime_confusion.png", dpi=130)
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for block, rows in pd.DataFrame(router_metrics).groupby("block"):
    axes[0].plot(rows["epoch"], rows["balanced_accuracy"], marker="o", label=f"Block {block}")
    axes[1].plot(rows["epoch"], rows["masked_rgb_mse"], marker="o", label=f"Block {block}")
axes[0].set(xlabel="Epoch (0 = new untrained router heads)", ylabel="Balanced regime accuracy", ylim=(0, 1))
axes[1].set(xlabel="Epoch", ylabel="Held-out masked RGB MSE (lower is better)")
axes[0].legend()
fig.savefig(run_dir / "training_validation.png", dpi=130)
plt.show()

## Video test: predictions choose the expert

Evaluate the same tunnel exits and three dawn/dusk clips. Tunnel clips with unknown time of day remain unscored; daytime tunnel frames keep the daytime clip label. The per-video accuracy uses coarse clip time-of-day metadata and does not invent framewise illumination labels. The full-frame predictions are **the actual gates**, not a separate classifier layered on top of unrelated expert dispatch.


In [ ]:
def directory_video_frames(spec: dict[str, Any]) -> Iterator[tuple[int, float, np.ndarray, float]]:
    """Read numbered extracted frames at their original capture rate."""
    path = Path(spec["path"])
    fps = float(spec.get("fps", 0))
    if not math.isfinite(fps) or fps <= 0:
        raise ValueError(f"{path}: frame directories require a positive source fps.")
    indexed = []
    for frame in path.iterdir():
        if frame.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        match = re.search(r"(\d+)$", frame.stem)
        if match is None:
            raise ValueError(f"Frame filename must end with its original frame index: {frame.name}")
        indexed.append((int(match.group(1)), frame))
    indexed.sort()
    if not indexed or len({i for i, _ in indexed}) != len(indexed):
        raise ValueError(f"Empty frame folder or duplicate numeric indices: {path}")
    for index, frame in indexed:
        seconds = (index - indexed[0][0]) / fps
        with Image.open(frame) as image:
            yield index, seconds, np.array(image.convert("RGB")), fps


def decoded_video_frames(spec: dict[str, Any]) -> Iterator[tuple[int, float, np.ndarray, float]]:
    """Decode every video frame in capture order and release the decoder on early exit."""
    path = Path(spec["path"])
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        capture.release()
        raise OSError(f"Cannot open video: {path}")
    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        if not math.isfinite(fps) or fps <= 0:
            raise ValueError(f"Video has no valid source FPS: {path}")
        index, previous_seconds = 0, -1.0
        while True:
            success, bgr = capture.read()
            if not success:
                break
            reported = float(capture.get(cv2.CAP_PROP_POS_MSEC)) / 1000
            seconds = reported if math.isfinite(reported) and reported > previous_seconds else index / fps
            if seconds <= previous_seconds:
                raise ValueError(f"Nonmonotonic timestamps in {path} at frame {index}.")
            previous_seconds = seconds
            yield index, seconds, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), fps
            index += 1
    finally:
        capture.release()


def video_frames(spec: dict[str, Any]) -> Iterator[tuple[int, float, np.ndarray, float]]:
    """Sample a continuous reader chronologically, without retaining full video in RAM."""
    reader = directory_video_frames(spec) if Path(spec["path"]).is_dir() else decoded_video_frames(spec)
    next_sample, count = 0.0, 0
    try:
        for index, seconds, rgb, fps in reader:
            if seconds + 1e-9 < next_sample:
                continue
            yield index, seconds, rgb, fps
            count += 1
            next_sample = seconds + sample_every_seconds
            if max_video_samples is not None and count >= max_video_samples:
                break
        if not count:
            raise ValueError(f"No frames decoded from {spec['path']}.")
    finally:
        reader.close()

In [ ]:
video_rows = []
with torch.no_grad():
    for spec in configured_videos:
        samples = []

        def process_video_batch(batch: list[tuple[int, float, np.ndarray, float]], clip_spec: dict[str, Any]) -> None:
            """Classify images with the real image gates; labels are used only afterward for scoring."""
            images = torch.stack([preprocess_rgb(item[2]) for item in batch]).to(device)
            errors = reconstruction_mse(images).cpu().numpy()
            for block, probability in method.encoder.routing(images).items():
                scores = probability[:, 0, :].cpu().numpy()
                for index, (frame_index, seconds, _rgb, _fps) in enumerate(batch):
                    predicted = int(scores[index].argmax())
                    video_rows.append(
                        {
                            "clip_id": clip_spec["clip_id"],
                            "frame_index": frame_index,
                            "seconds": seconds,
                            "block": block,
                            "predicted_regime": regime_names[predicted],
                            "selected_expert": predicted,
                            "true_regime": clip_spec["regime"] if clip_spec["regime"] in regime_to_id else None,
                            "brightness": float(images[index].mean()),
                            "masked_rgb_mse": float(errors[index]),
                            **{
                                f"probability_{name}": float(scores[index, expert])
                                for expert, name in enumerate(regime_names)
                            },
                        }
                    )

        for item in video_frames(spec):
            samples.append(item)
            if len(samples) == batch_size:
                process_video_batch(samples, spec)
                samples = []
        if samples:
            process_video_batch(samples, spec)
video_frame = pd.DataFrame(video_rows)
video_frame.to_csv(run_dir / "video_regime_routes.csv", index=False)
video_scores = []
for (clip, block), rows in video_frame.groupby(["clip_id", "block"]):
    labeled = rows.loc[rows["true_regime"].notna()]
    video_scores.append(
        {
            "clip_id": clip,
            "block": block,
            "frames": len(rows),
            "labeled_frames": len(labeled),
            "regime_accuracy": float((labeled["predicted_regime"] == labeled["true_regime"]).mean())
            if len(labeled)
            else None,
            "masked_rgb_mse": rows["masked_rgb_mse"].mean(),
        }
    )
pd.DataFrame(video_scores).to_csv(run_dir / "video_scores.csv", index=False)
show(pd.DataFrame(video_scores))

In [ ]:
colors = ["#e8a735", "#9964c0", "#375e9d"]
for clip_index, spec in enumerate(configured_videos):
    frame = video_frame.loc[video_frame["clip_id"] == spec["clip_id"]]
    fig, axes = plt.subplots(2 + 2 * len(moe_blocks), 1, figsize=(13, 14), sharex=True, constrained_layout=True)
    first = frame.loc[frame["block"] == moe_blocks[0]].sort_values("seconds")
    axes[0].plot(first["seconds"], first["brightness"], color="black", label="Mean RGB brightness")
    axes[0].set(ylabel="Brightness [0,1]", ylim=(0, 1))
    axes[0].legend()
    for block_index, block in enumerate(moe_blocks):
        probability_ax = axes[1 + 2 * block_index]
        route_ax = axes[2 + 2 * block_index]
        rows = frame.loc[frame["block"] == block].sort_values("seconds")
        for expert, name in enumerate(regime_names):
            probability_ax.plot(
                rows["seconds"], rows[f"probability_{name}"], color=colors[expert], label=f"E{expert}: {name}"
            )
        probability_ax.set(ylabel=f"Block {block} class probability", ylim=(0, 1))
        probability_ax.legend(ncol=3, fontsize=8)
        route_ax.step(
            rows["seconds"], rows["selected_expert"], color="black", where="post", label="Actual selected expert"
        )
        if spec["regime"] in regime_to_id:
            route_ax.axhline(regime_to_id[spec["regime"]], color="gray", linestyle="--", label="Clip regime label")
        route_ax.set_yticks(range(num_experts), [f"E{i}: {name}" for i, name in enumerate(regime_names)])
        route_ax.set(ylabel=f"Block {block} expert", ylim=(-0.2, num_experts - 0.8))
        route_ax.legend(fontsize=8)
    axes[-1].plot(first["seconds"], first["masked_rgb_mse"], color="#b04b4b")
    axes[-1].set_ylabel("Masked RGB MSE")
    for ax in axes:
        for seconds, _label in spec.get("markers", []):
            ax.axvline(seconds, color="gray", linestyle="--", alpha=0.5)
    axes[-1].set_xlabel("Seconds in the original continuous clip")
    fig.suptitle(spec["clip_id"] + ": image-predicted regime controls expert dispatch")
    fig.savefig(run_dir / f"video_{clip_index:02d}_regime_routes.png", dpi=130)
    plt.show()
print(f"Outputs: {run_dir}")
print("Model: supervised_router.pt; regime_metrics.csv; scene_probe_metrics.csv; video_regime_routes.csv")